# Feature Extraction for Random Forest

Extracts per-crown spectral and structural features from co-registered raster
datasets (RGB, CIR, CHM) by masking each crown polygon against the rasters and
computing summary statistics over the masked pixels.

**Features extracted per crown:**

| Feature | Source | Description |
|---|---|---|
| C_area | Polygon | Crown area in m² |
| R/G/B mean & std | RGB | Band reflectance statistics |
| NIR mean & std | CIR | Near-infrared reflectance statistics |
| NDVI mean & std | RGB + CIR | Normalised Difference Vegetation Index |
| CHM mean, std, max | CHM | Canopy height statistics |

The output shapefile (`tree_features_rf.shp`) is used directly as input for
Random Forest training and evaluation.

**Import libraries**

In [ ]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
import rasterio.mask

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR

**Define paths**

In [ ]:
# Crown polygons with species labels from RF_prepro_gt.ipynb
TREE_CROWNS_DIR = INTERIM_DIR / "random_forest_gt_prep/enschede_rf_ground_truth_v01.shp"

# Raw raster inputs
RGB_TILE_DIR = RAW_DIR / "rgb_tiles/RGB_34FN2.tif"
CIR_TILE_DIR = RAW_DIR / "cir_tiles/CIR_34FN2.tif"

# CHM resampled to 0.25 m to match RGB/CIR resolution
CHM_DIR = PROCESSED_DIR / "lidar/chm/chm_0p25.tif"

**Feature extraction function**

In [ ]:
def extract_features(tree_shp_dir, rgb_dir, cir_dir, chm_dir, output_path):
    """
    Extract per-crown spectral and structural features from raster datasets.

    For each crown polygon the function masks the RGB, CIR, and CHM rasters to
    the polygon extent using rasterio.mask.  Only pixels whose centres lie inside
    the polygon are included (all_touched=False).  Summary statistics (mean, std,
    max) are computed over the valid (non-masked) pixels using NumPy masked array
    operations.

    Args:
        tree_shp_dir:  Path to the crown polygon shapefile (from RF_prepro_gt.ipynb).
        rgb_dir:       Path to the RGB GeoTIFF (3 bands: R, G, B).
        cir_dir:       Path to the CIR GeoTIFF (band 1 = NIR).
        chm_dir:       Path to the CHM GeoTIFF resampled to 0.25 m.
        output_path:   Path where the feature shapefile will be saved.

    Returns:
        GeoDataFrame with one row per crown and extracted feature columns.

    Notes:
        - All rasters must share the same CRS; a ValueError is raised if not.
        - Crowns that overlap nodata cells in the CHM (arising from the resolution
          difference between the 0.5 m segmentation CHM and the 0.25 m feature
          CHM) produce NaN CHM features and should be dropped before modelling.
    """
    crowns = gpd.read_file(tree_shp_dir)

    # Initialise feature columns with NaN
    feature_cols = [
        "C_area",
        "R_mean", "G_mean", "B_mean",
        "R_std",  "G_std",  "B_std",
        "NIR_mean", "NIR_std",
        "NDVI_mean", "NDVI_std",
        "CHM_mean", "CHM_std", "CHM_max",
    ]
    for col in feature_cols:
        if col not in crowns.columns:
            crowns[col] = np.nan

    with rasterio.open(rgb_dir) as rgb,          rasterio.open(chm_dir) as chm,          rasterio.open(cir_dir) as cir:

        if not (rgb.crs == chm.crs == cir.crs):
            raise ValueError("CRS mismatch between rasters — reproject before running.")

        crowns = crowns.to_crs(rgb.crs)

        for idx, row in crowns.iterrows():
            geom = [row["geometry"]]

            # Mask each raster to the crown polygon.
            # filled=False returns a NumPy masked array; .mean() / .std() / .max()
            # automatically skip masked (outside-crown) pixels.
            rgb_arr, _ = rasterio.mask.mask(rgb, geom, crop=True, all_touched=False, filled=False)
            cir_arr, _ = rasterio.mask.mask(cir, geom, crop=True, all_touched=False, filled=False)
            chm_arr, _ = rasterio.mask.mask(chm, geom, crop=True, all_touched=False, filled=False)

            # RGB spectral features
            crowns.at[idx, "R_mean"] = float(rgb_arr[0].mean())
            crowns.at[idx, "G_mean"] = float(rgb_arr[1].mean())
            crowns.at[idx, "B_mean"] = float(rgb_arr[2].mean())
            crowns.at[idx, "R_std"]  = float(rgb_arr[0].std())
            crowns.at[idx, "G_std"]  = float(rgb_arr[1].std())
            crowns.at[idx, "B_std"]  = float(rgb_arr[2].std())

            # NIR spectral features (band 1 of CIR)
            crowns.at[idx, "NIR_mean"] = float(cir_arr[0].mean())
            crowns.at[idx, "NIR_std"]  = float(cir_arr[0].std())

            # NDVI: (NIR - R) / (NIR + R)
            R   = rgb_arr[0].astype(float)
            NIR = cir_arr[0].astype(float)
            ndvi = (NIR - R) / (NIR + R)
            crowns.at[idx, "NDVI_mean"] = float(ndvi.mean())
            crowns.at[idx, "NDVI_std"]  = float(ndvi.std())

            # CHM structural features
            crowns.at[idx, "CHM_mean"] = float(chm_arr[0].mean())
            crowns.at[idx, "CHM_std"]  = float(chm_arr[0].std())
            crowns.at[idx, "CHM_max"]  = float(chm_arr[0].max())

            # Crown area
            crowns.at[idx, "C_area"] = row.geometry.area

    # Drop crowns with missing CHM values (caused by nodata cells in the 0.25 m CHM)
    crowns = crowns.dropna(subset=["CHM_mean", "CHM_std"])

    crowns.to_file(output_path)
    return crowns

**Run feature extraction**

In [ ]:
features_rf = extract_features(
    tree_shp_dir = TREE_CROWNS_DIR,
    rgb_dir      = RGB_TILE_DIR,
    cir_dir      = CIR_TILE_DIR,
    chm_dir      = CHM_DIR,
    output_path  = PROCESSED_DIR / "rf_binary/enschede_rf_binary_features_v01.shp",
)

print(f"Crowns with features: {len(features_rf)}")
print(features_rf[["C_area", "R_mean", "NDVI_mean", "CHM_max", "is_allerge"]].head())